# B08-04 — BioCulture 연구 통합과 후속 배양 실험 설계

**목적:** B06–B08의 실제 측정 기반 분석과 회고적 최적화 성과를 재확인하고, *향후 수행할 실험*의 후보 선정·반복·대조·측정·평가·의사결정 프로토콜을 작성한다.

**중요:** 이 Notebook은 새 배양 실험을 수행하지 않습니다. 결과가 좋은 조성을 이미 알고 있는 회고적 분석과, 앞으로 실험할 조성을 구분합니다. 배지 성분의 인과 효과, 실험실 간 일반화, 독립적 검증은 입증되지 않았습니다.

실행 환경: Windows / Python 3.11 / `bioimage-ai` / pandas, numpy, matplotlib. 외부 인터넷이나 GPU는 필요하지 않습니다.

### 학습 목표
1. 검증된 1,025개 고유 조성과 B08 선택 이력의 정합성을 확인한다.
2. 전략 비교의 근거와 한계를 하나의 주장–증거 표로 작성한다.
3. 실제 후속 실험에 사용할 후보·대조군·반복·블라인딩·QC 프로토콜 초안을 만든다.
4. 새로운 측정이 이루어질 때만 확인할 수 있는 성공 조건을 분리한다.


## 1. 입력 파일, 버전, 과학적 범위

In [ ]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd=Path.cwd().resolve()
ROOT=next((p for p in [cwd,*cwd.parents] if (p/'results'/'bioculture'/'B08_03_selection_bias_audit_v1'/'B08_03_validation_report.json').is_file()),None)
if ROOT is None:
    raise FileNotFoundError('B08-03 감사 결과가 있는 bioimage-ai 프로젝트 폴더에서 실행하세요.')
BASE=ROOT/'results'/'bioculture'
CAND=BASE/'B08_00_candidate_design_v2'/'candidate_pool_measured_profiles.csv'
FEATURE=BASE/'B07_00_ml_data_audit_v2'/'medium_feature_audit.csv'
TRACE=BASE/'B08_02_bayesian_optimization_v1'/'bayesian_sequential_selection_trace.csv'
SUMMARY=BASE/'B08_02_bayesian_optimization_v1'/'bayesian_sequential_selection_summary.csv'
AUDIT=BASE/'B08_03_selection_bias_audit_v1'/'B08_03_validation_report.json'
ELIG=BASE/'B08_03_selection_bias_audit_v1'/'candidate_eligibility_audit.csv'
BIAS=BASE/'B08_03_selection_bias_audit_v1'/'selected_feature_bias_audit.csv'
OUT=BASE/'B08_04_prospective_experiment_design_v1'
for p in [CAND,FEATURE,TRACE,SUMMARY,AUDIT,ELIG,BIAS]:
    if not p.is_file(): raise FileNotFoundError(f'필수 입력 누락: {p}')
OUT.mkdir(parents=True,exist_ok=True)
print('Python:',sys.version.split()[0], '결과 폴더:',OUT)


## 2. 근거 자료 연결 및 완전성 재검사

In [ ]:
candidate=pd.read_csv(CAND,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
fi=pd.read_csv(FEATURE,encoding='utf-8-sig')
features=fi['feature'].astype(str).tolist()
trace=pd.read_csv(TRACE,encoding='utf-8-sig',dtype={'chosen_medium_profile_id':'string'})
summary=pd.read_csv(SUMMARY,encoding='utf-8-sig')
report=json.loads(AUDIT.read_text(encoding='utf-8'))
elig=pd.read_csv(ELIG,encoding='utf-8-sig')
bias=pd.read_csv(BIAS,encoding='utf-8-sig')
assert report['status']=='RETROSPECTIVE_SELECTION_AUDIT_COMPLETED_NOT_PROSPECTIVE_VALIDATION'
assert len(candidate)==1025 and candidate.medium_profile_id.is_unique
assert len(features)==44 and set(features).issubset(candidate.columns)
assert len(trace)==360 and len(summary)==18
assert {'K_median','r_median','n_rounds'}.issubset(candidate.columns)
assert {'target','seed','strategy','initial_best','final_best'}.issubset(summary.columns)
assert {'target','seed','strategy','step','chosen_medium_profile_id','revealed_value'}.issubset(trace.columns)
for t in ['K_median','r_median']:
    q=trace.loc[trace.target.eq(t)]
    truth=candidate.set_index('medium_profile_id')[t]
    assert np.allclose(q.chosen_medium_profile_id.map(truth).astype(float),q.revealed_value.astype(float),atol=1e-9)
assert trace.groupby(['target','seed','strategy']).chosen_medium_profile_id.nunique().eq(20).all()
print('입력 정합성 통과: 조성',len(candidate),'성분',len(features),'선택 기록',len(trace))
print('목표값 존재 조성:',{t:int(candidate[t].notna().sum()) for t in ['K_median','r_median']})


## 3. 관측 사실·회고적 평가·향후 실험 주장을 분리하기

In [ ]:
claims=pd.DataFrame([
 {'claim':'원본 측정 기반 고유 조성 및 입력 성분 구성','evidence':'B07-00 / B08-00 데이터 감사','status':'observed_data_structure','limitation':'조성 식별은 명목 농도 기준이며 실험기록과 독립 비교는 별도'},
 {'claim':'Random/RF/GP_EI의 기존 측정값 발견 비교','evidence':'B08-01~03 동일 시드·20회 회고적 선택 이력','status':'retrospective_only','limitation':'후보 자격에 과거 목표값 존재 여부를 사용'},
 {'claim':'라운드 전환에서 예측 성능 유지','evidence':'B07-03 라운드 보류 평가','status':'not_supported','limitation':'라운드별 큰 예측 편향과 조성 차이 존재'},
 {'claim':'특정 배지 성분의 인과효과','evidence':'B07-01 상관·B07-04 중요도','status':'not_supported','limitation':'교란·성분 공변화 통제 실험 없음'},
 {'claim':'새 배양 실험에서 선택 전략의 우월성','evidence':'새 실험 측정값 없음','status':'requires_new_experiment','limitation':'전향적 측정 및 사전 지정된 평가 필요'}
])
print(claims[['claim','status']].to_string(index=False))


## 4. 향후 실제 실험에서 확인할 질문과 후보 구성의 원칙

실험의 목적은 **한 번 더 높은 값을 찾았다**는 주장보다 **재현 가능한 배지 조성–성장 지표 관계**를 평가하는 것이다. 과거 최고값을 이미 알고 선택하는 경우에는 확인 실험(confirmatory study)으로, 실제로 미측정 배지를 새로 제안할 경우에는 탐색 실험(discovery study)으로 각각 기록한다.

후보에는 각 목표의 기존 관측 상위 조성, 두 목표의 절충 조성, 목표값이 낮은 대조 후보, 여러 라운드에서 측정된 교차 확인 후보를 포함한다. **후보군 선정은 기존 측정값에 기반한 목적적 선정**이므로 무작위 추출 결과가 아니다.

In [ ]:
work=candidate.copy()
for t in ['K_median','r_median']:
    work[t]=pd.to_numeric(work[t],errors='coerce')
# 목표를 동시에 만족하는 실측 조성에서 두 목표의 관측 백분위 평균을 계산
both=work.loc[work[['K_median','r_median']].notna().all(axis=1)].copy()
for t in ['K_median','r_median']:
    both[t+'_pct_rank']=both[t].rank(pct=True)
both['balanced_rank_score']=(both['K_median_pct_rank']+both['r_median_pct_rank'])/2
choices=[]
def add_candidates(df, rationale, n=3):
    for _,r in df.head(n).iterrows():
        choices.append({'medium_profile_id':str(r['medium_profile_id']), 'nomination_reason':rationale,
                        'historical_K_median':float(r['K_median']), 'historical_r_median':float(r['r_median']),
                        'historical_n_rounds':int(r['n_rounds'])})
add_candidates(both.sort_values('K_median',ascending=False),'historical_high_K')
add_candidates(both.sort_values('r_median',ascending=False),'historical_high_r')
add_candidates(both.sort_values('balanced_rank_score',ascending=False),'historical_balanced')
add_candidates(both.sort_values('balanced_rank_score'),'historical_low_reference',2)
add_candidates(both.loc[both.n_rounds.gt(1)].sort_values('balanced_rank_score',ascending=False),'cross_round_reference',3)
nominees=pd.DataFrame(choices).drop_duplicates('medium_profile_id').reset_index(drop=True)
nominees.insert(0,'nomination_index',range(1,len(nominees)+1))
nominees['new_experiment_observed']=False
nominees['approval_status']='PROPOSED_NOT_WETLAB_APPROVED'
print('제안 후보 수:',len(nominees))
print(nominees[['nomination_index','medium_profile_id','nomination_reason','historical_K_median','historical_r_median']].to_string(index=False))


## 5. 실제 실험 프로토콜 초안 — 확정 전에 실험 책임자가 검토할 사항

아래는 **실행 지시가 아니라 계획 초안**이다. 원 논문의 실험 조건, 사용 균주·플레이트·리더기와 배지 제조 가능성, 안전 규정 및 연구 목적에 맞게 책임자가 확정해야 한다. 실험 라운드를 가로지르는 비교에서는 공통 참조 배지를 반드시 반복하여 측정해야 한다.

- 사전 정의: 1차 목표는 `K` 계열 또는 `r` 계열 중 하나로 우선 확정하고 측정·계산 방법과 단위를 문서화한다. 둘 다 다루면 다중목표 평가 규칙을 사전 지정한다.
- 대조군: 기존 연구의 표준 배지, 성장 음성·장비 blank, 기존 결과 재현용 참조 조성. 대조군은 실제 실험실에서 확정한다.
- 반복: 각 후보를 독립적으로 준비한 배양 단위로 여러 회 측정하며, 플레이트 내 기술 반복과 서로 다른 날·배치의 생물학적 반복을 구분하여 기록한다. 아래 수량은 실행 전 합의할 설계 예시일 뿐이다.
- 무작위화: 플레이트 위치를 무작위 배치하고 측정자가 후보의 기존 성과를 모르게 블라인딩한다.
- 라운드: 가능하면 서로 다른 날짜·배치에 후보군과 공통 참조를 모두 포함해 라운드–조성 교란을 완화한다.
- 원자료: OD600 시간축·원본 측정값, blank 보정, 배지 제조 기록, 성장 실패/결측 사유를 모두 보존한다.
- 승인: 실제 실험 수행 여부, 규모, 표준운영절차는 실험 책임자가 검토·승인한다.

In [ ]:
protocol={
 'study_type':'PROPOSED_PROSPECTIVE_CONFIRMATION_NOT_EXECUTED',
 'source':'B06-B08 E. coli retrospective project',
 'candidate_count':int(len(nominees)),
 'primary_endpoint':'TO_BE_SPECIFIED_BEFORE_EXPERIMENT',
 'endpoint_method_units':'ORIGINAL_K_R_DEFINITIONS_AND_OD600_PROCESSING_MUST_BE_VERIFIED',
 'control_requirements':['reference_medium','instrument_blank','reproducibility_reference'],
 'replication_example_not_final':{'independent_days':3,'independent_cultures_per_day':3},
 'randomize_plate_positions':True,'blind_historical_ranking_to_operator':True,
 'record_batch_instrument_medium_preparation':True,
 'pre_register_exclusion_and_qc_criteria':True,
 'approval_required':True,
 'data_to_record':['medium_profile_id','exact_concentrations','strain','date','plate','well','replicate_type','raw_time_OD600','blank','QC_failure_reason'],
 'disallowed_claims':['new experiment completed','optimized medium confirmed','causal ingredient effect confirmed']
}
print(json.dumps(protocol,ensure_ascii=False,indent=2))


## 6. 평가 지표·의사결정 규칙 및 새 실험 이후의 검증 게이트

분석의 최소 원칙: (1) 독립 배양 단위를 우선 분석하고, (2) 같은 날·플레이트의 중첩 측정을 독립 표본처럼 취급하지 않으며, (3) 실험 라운드를 명시적으로 포함하고, (4) 실패/누락을 숨기지 않는다.

새 실험 데이터가 도착한 이후에만 다음을 비교할 수 있다: 계획된 참조 배지 대비 후보의 새로운 실측 성장 지표, 반복 간 변동, 새로운 라운드의 재현성, 기존 회고적 순위와 새로운 측정 순위의 관계.

아래 게이트는 해당 연구의 결과가 아직 측정되지 않았음을 분명히 기록한다.

In [ ]:
gates=pd.DataFrame([
 {'gate':'G1','criterion':'candidate chemistry and safety approval','status':'PENDING_HUMAN_REVIEW'},
 {'gate':'G2','criterion':'protocol endpoints controls randomization preregistered','status':'PENDING_HUMAN_REVIEW'},
 {'gate':'G3','criterion':'prospective raw OD600 and metadata recorded','status':'NO_NEW_DATA'},
 {'gate':'G4','criterion':'QC and reproducibility across independent batches evaluated','status':'NOT_TESTED'},
 {'gate':'G5','criterion':'prospective comparison against predeclared references','status':'NOT_TESTED'}
])
assert gates.status.ne('PASS').all()
print(gates.to_string(index=False))


## 7. 최종 출력, 재현성 감사 및 실행 상태

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
outputs={
 'evidence_claim_matrix.csv':claims,
 'prospective_candidate_nominations.csv':nominees,
 'prospective_validation_gates.csv':gates,
}
for name,df in outputs.items():
    p=OUT/name;df.to_csv(p,index=False,encoding='utf-8-sig')
    test=pd.read_csv(p,encoding='utf-8-sig')
    assert len(test)==len(df) and list(test.columns)==list(df.columns)
protocol_path=OUT/'proposed_experiment_protocol.json'
protocol_path.write_text(json.dumps(protocol,ensure_ascii=False,indent=2),encoding='utf-8')
summary_report={
 'status':'B08_04_INTEGRATED_PROSPECTIVE_DESIGN_NOT_EXECUTED',
 'n_historical_medium_profiles':int(len(candidate)),
 'n_historical_features':len(features),
 'n_historical_selection_records':int(len(trace)),
 'n_proposed_medium_profiles':int(len(nominees)),
 'prospective_raw_measurements_available':False,
 'approval_obtained':False,
 'note':'All candidate historical results are pre-existing observational data, not new lab results.',
}
rep=OUT/'B08_04_validation_report.json'
rep.write_text(json.dumps(summary_report,ensure_ascii=False,indent=2),encoding='utf-8')
inputs=[CAND,FEATURE,TRACE,SUMMARY,AUDIT,ELIG,BIAS]
written=[OUT/k for k in outputs]+[protocol_path,rep]
manifest=pd.DataFrame([{'path':str(p.relative_to(ROOT)),'role':'input' if p in inputs else 'output','bytes':p.stat().st_size,'sha256':sha256(p)} for p in inputs+written])
manifest.to_csv(OUT/'input_output_sha256_manifest.csv',index=False,encoding='utf-8-sig')
print(json.dumps(summary_report,ensure_ascii=False,indent=2))
print('출력 파일:',len(written)+1)


## 8. 연습문제 (정답·풀이 코드는 별도 Word 해설서에 수록)

1. 기존 결과가 높은 배지를 실제 후속 실험에서 다시 측정해야 하는 이유를 설명하라.
2. `nominees`에서 동일 배지 조성이 여러 기준에 의해 지명되는 경우를 조사하고, 지명 근거가 여러 개인 후보를 보존하는 방법을 제안하라.
3. `historical_n_rounds>1` 조성의 개수와 그 의미를 계산하고, 이를 독립 반복 수와 혼동하면 안 되는 이유를 설명하라.
4. `prospective_validation_gates.csv`에서 아직 미충족된 항목을 자동으로 나열하는 코드를 작성하라.
5. `K_median`과 `r_median`의 절충 순위가 각 단일 목표 1위와 다를 수 있음을 실제 후보 표로 비교하고, 다중목표 최적화의 한계를 설명하라.

**마무리:** B08-04는 교육용 사례의 분석 결과와 미래 실험 제안서를 연결하지만, 전향적 실험 검증 완료를 뜻하지 않는다. 다음 단계는 B08 전체 감사와 BioCulture 교재 Part III 집필 계획 검토이다.
